In [8]:
 
# LIGHTCONE GENERATION - BOX SIZE SCAN
# py21cmfast v3.4
# Fixed Resolution (HII_DIM), varying BOX_LEN from 400 to 2000 Mpc
# =============================================================================

# =============================================================================
# CELL 1: Imports and Setup
# =============================================================================

import numpy as np
import matplotlib as mpl

import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import matplotlib.pyplot as plt

import py21cmfast as p21c
from py21cmfast import plotting

import os
from datetime import datetime

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Create Output Directory for Plots
# =============================================================================

plot_dir = "BOX_SIZE_SCAN_3Feb2026/plots"

# Create directory if it doesn't exist
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")

print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")



# =============================================================================
# CELL 1c: Define Parameters
# =============================================================================


# **FIXED: CELL SIZE (RESOLUTION IN MPC)**
CELL_SIZE_MPC = 400 / 128  # ~3.125 Mpc (or choose your desired fixed resolution)

# **SCAN: BOX_LEN (400 to 2000 Mpc in steps of 400)**
BOX_LEN_VALUES = np.arange(200, 1401, 400)  # [400, 800, 1200, 1600, 2000] Mpc

# **COMPUTED: HII_DIM scales with BOX_LEN to maintain fixed cell size**
HII_DIM_VALUES = (BOX_LEN_VALUES / CELL_SIZE_MPC).astype(int)

# Redshift range for the lightcone
z_min = 5.0
z_max = 20.0

print(f"\n=== PARAMETER SCAN SETUP ===")
print(f"Fixed CELL_SIZE = {CELL_SIZE_MPC:.3f} Mpc")
print(f"\nScanning BOX_LEN:")
print(f"  Number of values: {len(BOX_LEN_VALUES)}")
print(f"  Range: {BOX_LEN_VALUES.min():.0f} Mpc → {BOX_LEN_VALUES.max():.0f} Mpc")
print(f"  Step size: {BOX_LEN_VALUES[1] - BOX_LEN_VALUES[0]:.0f} Mpc")
print(f"  Values: {BOX_LEN_VALUES}")
print(f"\nCorresponding HII_DIM values: {HII_DIM_VALUES}")
print(f"\nTotal simulations: {len(BOX_LEN_VALUES)}")

# ... rest of your code ...

# Print resolution info for each box size
print("\n=== RESOLUTION DETAILS ===")
print(f"{'BOX_LEN [Mpc]':<15} {'HII_DIM':<10} {'Cell Size [Mpc]':<20} {'Cell Size [kpc]':<15}")
print("-" * 70)
for box_len, hii_dim in zip(BOX_LEN_VALUES, HII_DIM_VALUES):
    cell_size_mpc = box_len / hii_dim
    cell_size_kpc = cell_size_mpc * 1000
    print(f"{box_len:<15.0f} {hii_dim:<10} {cell_size_mpc:<20.3f} {cell_size_kpc:<15.1f}")



py21cmfast version: 3.3.1
Created directory: BOX_SIZE_SCAN_3Feb2026/plots
All plots will be saved to: /user1/swanith/BOX_SIZE_SCAN_3Feb2026/plots

=== PARAMETER SCAN SETUP ===
Fixed CELL_SIZE = 3.125 Mpc

Scanning BOX_LEN:
  Number of values: 4
  Range: 200 Mpc → 1400 Mpc
  Step size: 400 Mpc
  Values: [ 200  600 1000 1400]

Corresponding HII_DIM values: [ 64 192 320 448]

Total simulations: 4

=== RESOLUTION DETAILS ===
BOX_LEN [Mpc]   HII_DIM    Cell Size [Mpc]      Cell Size [kpc]
----------------------------------------------------------------------
200             64         3.125                3125.0         
600             192        3.125                3125.0         
1000            320        3.125                3125.0         
1400            448        3.125                3125.0         


In [5]:
# =============================================================================
# CELL 1d: Define Plotting Utilities
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib as mpl

PDF_STYLE = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 30,
    'axes.labelsize': 29,
    'axes.titlesize': 40,
    'xtick.labelsize': 30,
    'ytick.labelsize': 30,
    'legend.fontsize': 20,
    'figure.titlesize': 28,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.minor.width': 0.8,
    'ytick.minor.width': 0.8,
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.8,
    'lines.markersize': 5,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
}

PNG_STYLE = {
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 15,
    'axes.labelsize': 25,
    'axes.titlesize': 18,
    'xtick.labelsize': 25,
    'ytick.labelsize': 25,
    'legend.fontsize': 25,
    'figure.titlesize': 15,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.5,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
}

def save_pdf_png(plot_func, plot_dir, plot_name, title=None):
    """Save plot as both PDF and PNG"""
    with mpl.rc_context(PDF_STYLE):
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        plot_func(ax)
        ax.set_title("")
        fig.savefig(f"{plot_dir}/{plot_name}.pdf")
        plt.close(fig)

    with mpl.rc_context(PNG_STYLE):
        fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
        plot_func(ax)
        if title is not None:
            ax.set_title(title, fontweight='bold')
        fig.savefig(f"{plot_dir}/{plot_name}.png")
        plt.close(fig)

def redshift_to_time(z, H0=67.4, Om0=0.315):
    """Convert redshift to time since Big Bang in Gyr"""
    from astropy.cosmology import FlatLambdaCDM
    import astropy.units as u
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0)
    return cosmo.age(z).to(u.Gyr).value

def make_lightcone_plotter(
    lightcone,
    field,
    cmap=None,
    clim=None,
    cbar_label=None,
    labelsize=16,
    axlabelsize=20,
    user_params=None,
    add_time_axis=False,
):
    """Create a lightcone plotting function"""
    def _plot(ax):
        import numpy as np
        fig = ax.figure
        
        # Plot the lightcone
        plotting.lightcone_sliceplot(lightcone, field, ax=ax, fig=fig)
        im = ax.images[0]
        
        # Apply colormap and limits
        if cmap is not None:
            im.set_cmap(cmap)
        if clim is not None:
            im.set_clim(*clim)
        
        # Y-axis label
        if user_params is not None:
            ax.set_ylabel(r'y-axis [Mpc]', fontsize=axlabelsize) #[$h^{-1}$ cMpc]
        
        # Tick formatting
        ax.tick_params(axis='both', which='major', labelsize=labelsize)
        ax.xaxis.label.set_size(axlabelsize)
        ax.yaxis.label.set_size(axlabelsize)
        
        # Add time axis on top if requested
        ax2 = None
        if add_time_axis:
            ax2 = ax.twiny()
            xlim = ax.get_xlim()
            z_min, z_max = xlim
            t_min = redshift_to_time(z_max)
            t_max = redshift_to_time(z_min)
            
            ax2.set_xlim(t_min, t_max)
            ax2.set_xlabel('Time Since Big Bang [Gyr]', fontsize=axlabelsize)
            
            # Smart tick spacing
            time_range = t_max - t_min
            if time_range < 2:
                tick_spacing = 0.2
            elif time_range < 5:
                tick_spacing = 0.5
            elif time_range < 10:
                tick_spacing = 1.0
            else:
                tick_spacing = 2.0
            
            first_tick = np.ceil(t_min / tick_spacing) * tick_spacing
            last_tick = np.floor(t_max / tick_spacing) * tick_spacing
            t_ticks = np.arange(first_tick, last_tick + tick_spacing/2, tick_spacing)
            
            if len(t_ticks) > 8:
                t_ticks = t_ticks[::2]
            
            ax2.set_xticks(t_ticks)
            ax2.set_xticklabels([f'{t:.1f}' for t in t_ticks])
            ax2.tick_params(axis='x', which='major', labelsize=labelsize, direction='in')
            ax2.tick_params(axis='x', which='minor', direction='in')
            ax2.minorticks_on()
        
        # Format colorbar
        for cax in fig.axes:
            if cax is ax:
                continue
            if ax2 is not None and cax is ax2:
                continue
            cax.tick_params(labelsize=labelsize)
            cax.xaxis.label.set_size(axlabelsize)
            if cbar_label is not None:
                cax.set_xlabel(cbar_label)
    
    return _plot

print("✓ Simple lightcone plotter loaded")



✓ Simple lightcone plotter loaded


In [6]:
# =============================================================================
# CELL 2: Run Lightcone Simulations - BOX_LEN Scan
# =============================================================================

import time

print("\n" + "="*70)
print("RUNNING BOX_LEN SCAN (FIXED RESOLUTION)")
print("="*70)

# Create main cache directory
main_cache_dir = "BOX_SIZE_SCAN_3Feb2026/cache"
if not os.path.exists(main_cache_dir):
    os.makedirs(main_cache_dir)
    print(f"Created cache directory: {main_cache_dir}")
else:
    print(f"Cache directory exists: {main_cache_dir}")

# Dictionary to store lightcone results
lightcones = {}

# Track timing
scan_start_time = time.time()

for idx, (box_len, hii_dim) in enumerate(zip(BOX_LEN_VALUES, HII_DIM_VALUES)):
    sim_start_time = time.time()
    
    print(f"\n{'='*70}")
    print(f"SIMULATION {idx+1}/{len(BOX_LEN_VALUES)}")
    print(f"BOX_LEN = {box_len:.0f} Mpc")
    print(f"HII_DIM = {hii_dim}")
    print(f"Cell size = {box_len/hii_dim:.3f} Mpc = {box_len/hii_dim*1000:.1f} kpc")
    print(f"{'='*70}")
    
    # Define user parameters for this box size
    user_params = p21c.UserParams(
        HII_DIM=hii_dim,
        BOX_LEN=box_len,
        USE_INTERPOLATION_TABLES=True,
        N_THREADS=32
    )
    
    print(f"Redshift range: z = {z_min} → {z_max}")
    print(f"Resolution: {user_params.HII_DIM}³ cells")
    
    # Create subdirectory for this box size
    cache_subdir = f"{main_cache_dir}/BOX{box_len:.0f}_DIM{hii_dim}"
    
    # ← NEW: Check if lightcone exists in cache
    lightcone_file = f"{cache_subdir}/LightCone_z{z_min:.2f}_z{z_max:.2f}.h5"
    
    if os.path.exists(lightcone_file):
        print(f"\n✓ Found cached lightcone: {lightcone_file}")
        print(f"  Loading from cache instead of recomputing...")
        try:
            # Load existing lightcone
            from py21cmfast import LightCone
            lightcone = LightCone.read(lightcone_file)
            lightcones[box_len] = lightcone
            
            sim_time = time.time() - sim_start_time
            print(f"  Load time: {sim_time:.2f} seconds")
            print(f"  Shape: {lightcone.brightness_temp.shape}")
            print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
                  f"{lightcone.lightcone_redshifts.max():.2f}]")
            
            # Skip to next iteration
            continue
            
        except Exception as e:
            print(f"  ✗ Failed to load cached lightcone: {e}")
            print(f"  Will recompute...")
    
    # Run lightcone simulation (only if not cached)
    try:
        lightcone = p21c.run_lightcone(
            redshift=z_min,
            max_redshift=z_max,
            lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
            user_params=user_params,
            random_seed=37,
            direc=cache_subdir,
            write=True  # ← Make sure to save!
        )
        
        # Store the lightcone
        lightcones[box_len] = lightcone
        

        sim_time = time.time() - sim_start_time
        
        print(f"\n✓ Simulation complete!")
        print(f"  Time: {sim_time/60:.2f} minutes")
        print(f"  Cache: {cache_subdir}")
        print(f"  Shape: {lightcone.brightness_temp.shape}")
        print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
              f"{lightcone.lightcone_redshifts.max():.2f}]")
        
        # Quick reionization stats
        z_nodes = lightcone.node_redshifts[::-1]
        x_e_nodes = 1.0 - lightcone.global_xH[::-1]
        
        try:
            idx_10 = np.argmin(np.abs(x_e_nodes - 0.1))
            idx_50 = np.argmin(np.abs(x_e_nodes - 0.5))
            idx_90 = np.argmin(np.abs(x_e_nodes - 0.9))
            
            z_10 = z_nodes[idx_10]
            z_50 = z_nodes[idx_50]
            z_90 = z_nodes[idx_90]
            delta_z = z_10 - z_90
            
            print(f"  Quick stats:")
            print(f"    z(10% ionized) = {z_10:.2f}")
            print(f"    z(50% ionized) = {z_50:.2f}")
            print(f"    z(90% ionized) = {z_90:.2f}")
            print(f"    Δz (10%→90%) = {delta_z:.2f}")
        except:
            print(f"  Could not compute reionization stats")
        
        # Progress estimate
        
        elapsed = time.time() - scan_start_time
        avg_time_per_sim = elapsed / (idx + 1)
        remaining_sims = len(BOX_LEN_VALUES) - (idx + 1)
        eta_minutes = (remaining_sims * avg_time_per_sim) / 60
        
        print(f"\n  Progress: {idx+1}/{len(BOX_LEN_VALUES)} ({100*(idx+1)/len(BOX_LEN_VALUES):.1f}%)")
        print(f"  Average time per sim: {avg_time_per_sim/60:.2f} min")
        print(f"  ETA: {eta_minutes:.1f} minutes (~{eta_minutes/60:.2f} hours)")
        
    except Exception as e:
        print(f"\n✗ Simulation FAILED!")
        print(f"  Error: {e}")
        lightcones[box_len] = None

total_time = time.time() - scan_start_time

print(f"\n{'='*70}")
print("ALL SIMULATIONS COMPLETE")
print(f"Total time: {total_time/60:.2f} minutes ({total_time/3600:.2f} hours)")
print(f"Successful simulations: {sum(1 for lc in lightcones.values() if lc is not None)}/{len(BOX_LEN_VALUES)}")
print("="*70)


RUNNING BOX_LEN SCAN (FIXED RESOLUTION)
Cache directory exists: BOX_SIZE_SCAN_3Feb2026/cache

SIMULATION 1/4
BOX_LEN = 200 Mpc
HII_DIM = 64
Cell size = 3.125 Mpc = 3125.0 kpc
Redshift range: z = 5.0 → 20.0
Resolution: 64³ cells

✓ Simulation complete!
  Time: 0.44 minutes
  Cache: BOX_SIZE_SCAN_3Feb2026/cache/BOX200_DIM64
  Shape: (64, 64, 965)
  Redshift range: [5.00, 20.06]
  Quick stats:
    z(10% ionized) = 10.76
    z(50% ionized) = 7.74
    z(90% ionized) = 6.17
    Δz (10%→90%) = 4.59

  Progress: 1/4 (25.0%)
  Average time per sim: 0.54 min
  ETA: 1.6 minutes (~0.03 hours)

SIMULATION 2/4
BOX_LEN = 600 Mpc
HII_DIM = 192
Cell size = 3.125 Mpc = 3125.0 kpc
Redshift range: z = 5.0 → 20.0
Resolution: 192³ cells

✓ Simulation complete!
  Time: 4.38 minutes
  Cache: BOX_SIZE_SCAN_3Feb2026/cache/BOX600_DIM192
  Shape: (192, 192, 965)
  Redshift range: [5.00, 20.06]
  Quick stats:
    z(10% ionized) = 10.76
    z(50% ionized) = 7.74
    z(90% ionized) = 6.17
    Δz (10%→90%) = 4.59

 

In [ ]:
# =============================================================================
# NOT FOR REPORTs
# CELL 2b: Plot Lightcones (All Box Sizes Stacked)
# =============================================================================
print("\n" + "="*70)
print("GENERATING STACKED LIGHTCONE PLOTS")
print("="*70)

# We'll plot all box sizes in the scan
print(f"Plotting all {len(BOX_LEN_VALUES)} box sizes")

# Create colormap for labeling
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

# Fields to plot
fields_to_plot = [
    ('brightness_temp', '21cm Brightness Temperature', 'EoR'),
    ('xH_box', 'Neutral Fraction (xHI)', 'viridis'),
    ('density', 'Overdensity δ', 'magma'),
    ('velocity', 'Line-of-Sight Velocity', 'RdBu_r')
]

for field_name, field_title, field_cmap in fields_to_plot:
    print(f"\nPlotting {field_name}...")
    
    # Create figure with one subplot per box size
    fig, axes = plt.subplots(len(BOX_LEN_VALUES), 1, 
                            figsize=(14, 4*len(BOX_LEN_VALUES)), 
                            constrained_layout=True)
    
    if len(BOX_LEN_VALUES) == 1:
        axes = [axes]
    
    for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
        if box_len not in lightcones or lightcones[box_len] is None:
            ax.text(0.5, 0.5, f'BOX_LEN = {box_len:.0f} Mpc\nSimulation Failed', 
                   ha='center', va='center', fontsize=16, color='red',
                   transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
            
        lightcone = lightcones[box_len]
        
        # Plot using py21cmfast's built-in plotter
        plotting.lightcone_sliceplot(lightcone, field_name, ax=ax, fig=fig)
        
        # Change colormap
        im = ax.images[0]
        im.set_cmap(field_cmap)
        
        # Get color for this box size
        color = cmap(norm(box_len))
        
        # Calculate cell size
        cell_size_mpc = box_len / HII_DIM_FIXED
        cell_size_kpc = cell_size_mpc * 1000
        
        # Add label with box info
        ax.text(0.02, 0.98, 
               f'BOX = {box_len:.0f} Mpc  |  Cell = {cell_size_mpc:.2f} Mpc ({cell_size_kpc:.1f} kpc)', 
               transform=ax.transAxes, fontsize=13, fontweight='bold',
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor=color, alpha=0.7, 
                        edgecolor='black', linewidth=2))
        
        # Add box size on right side as well
        ax.text(0.98, 0.98, 
               f'{box_len:.0f} Mpc', 
               transform=ax.transAxes, fontsize=14, fontweight='bold',
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Save
    plot_name = f"{field_name}_lightcone_boxsize_stack"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    
    fig.suptitle(f'{field_title} - Box Size Scan (HII_DIM={HII_DIM_FIXED})', 
                fontsize=22, fontweight='bold', y=0.995)
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"  ✓ Saved: {plot_name}")
    plt.close(fig)

print("\n✓ STACKED LIGHTCONE PLOTTING COMPLETE!")


GENERATING STACKED LIGHTCONE PLOTS
Plotting all 5 box sizes

Plotting brightness_temp...
  ✓ Saved: brightness_temp_lightcone_boxsize_stack

Plotting xH_box...
  ✓ Saved: xH_box_lightcone_boxsize_stack

Plotting density...
  ✓ Saved: density_lightcone_boxsize_stack

Plotting velocity...
  ✓ Saved: velocity_lightcone_boxsize_stack

✓ STACKED LIGHTCONE PLOTTING COMPLETE!


In [ ]:
# =============================================================================
#NOT FOR REPORTs
# CELL 3: 2D SLICES AT z = 8 (ALL BOX SIZES)
# =============================================================================

print("\n" + "="*70)
print("GENERATING 2D SLICES AT z = 8.0")
print("="*70)

target_z = 8.0

# Use all box sizes
print(f"Using all {len(BOX_LEN_VALUES)} BOX_LEN values for slices")

fields_info = [
    ('brightness_temp', '21cm Brightness Temperature [mK]', 'EoR'),
    ('xH_box', 'Neutral Fraction (xHI)', 'viridis'),
    ('density', 'Overdensity δ', 'magma'),
    ('velocity', 'Line-of-Sight Velocity [km/s]', 'RdBu_r'),
]

for field_name, field_label, cmap_name in fields_info:
    print(f"\nProcessing {field_name}...")
    
    # Create figure with all box sizes
    fig, axes = plt.subplots(1, len(BOX_LEN_VALUES), 
                            figsize=(5*len(BOX_LEN_VALUES), 5.5), 
                            constrained_layout=True)
    
    if len(BOX_LEN_VALUES) == 1:
        axes = [axes]
    
    # First pass: find global min/max for consistent colorbar
    vmin_global = np.inf
    vmax_global = -np.inf
    slices_data = []
    
    for box_len in BOX_LEN_VALUES:
        if box_len not in lightcones or lightcones[box_len] is None:
            slices_data.append((None, None, box_len))
            continue
            
        lightcone = lightcones[box_len]
        z_values = lightcone.lightcone_redshifts
        closest_idx = np.argmin(np.abs(z_values - target_z))
        actual_z = z_values[closest_idx]
        
        field_data = getattr(lightcone, field_name)
        slice_2d = field_data[:, :, closest_idx]
        
        slices_data.append((slice_2d, actual_z, box_len))
        
        vmin_global = min(vmin_global, slice_2d.min())
        vmax_global = max(vmax_global, slice_2d.max())
    
    # Create rainbow colormap for box labels
    cmap_rainbow = mpl.cm.rainbow
    norm_rainbow = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())
    
    # Second pass: plot with consistent colorbar
    for idx, (data_tuple, ax) in enumerate(zip(slices_data, axes)):
        slice_2d, actual_z, box_len = data_tuple
        
        if slice_2d is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                   transform=ax.transAxes, fontsize=16)
            ax.set_title(f'BOX={box_len:.0f} Mpc', fontsize=14)
            continue
        
        im = ax.imshow(slice_2d.T, origin='lower', cmap=cmap_name,
                      vmin=vmin_global, vmax=vmax_global,
                      extent=[0, box_len, 0, box_len],
                      aspect='auto')
        
        ax.set_xlabel('Comoving Distance [Mpc]', fontsize=12)
        ax.set_ylabel('Comoving Distance [Mpc]', fontsize=12)
        
        # Get rainbow color for this box size
        color_label = cmap_rainbow(norm_rainbow(box_len))
        
        # Calculate cell size
        cell_size_mpc = box_len / HII_DIM_FIXED
        cell_size_kpc = cell_size_mpc * 1000
        
        ax.set_title(f'BOX={box_len:.0f} Mpc\nCell={cell_size_mpc:.2f} Mpc ({cell_size_kpc:.0f} kpc)\nz={actual_z:.2f}', 
                    fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor=color_label, alpha=0.6, 
                             edgecolor='black', linewidth=1.5))
        
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=10)
    
    # Overall title
    fig.suptitle(f'{field_label} at z ≈ {target_z} (HII_DIM={HII_DIM_FIXED})', 
                fontsize=18, fontweight='bold')
    
    # Save
    plot_name = f"{field_name}_slice_z{int(target_z)}_boxsize_all"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"  ✓ Saved: {plot_name}")
    print(f"    Range: [{vmin_global:.3f}, {vmax_global:.3f}]")
    
    plt.close(fig)

print("\n✓ 2D SLICE PLOTTING COMPLETE!")

# =============================================================================
# CELL 3b: Print Statistics for 2D Slices
# =============================================================================

print("\n" + "="*70)
print("2D SLICE STATISTICS AT z = 8")
print("="*70)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    z_values = lightcone.lightcone_redshifts
    closest_idx = np.argmin(np.abs(z_values - target_z))
    actual_z = z_values[closest_idx]
    
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    print(f"\nBOX_LEN = {box_len:.0f} Mpc (Cell = {cell_size_mpc:.2f} Mpc = {cell_size_kpc:.0f} kpc) at z = {actual_z:.2f}:")
    
    for field_name, field_label, _ in fields_info:
        field_data = getattr(lightcone, field_name)
        slice_2d = field_data[:, :, closest_idx]
        
        print(f"  {field_name:15s}: min={slice_2d.min():10.3e}, "
              f"max={slice_2d.max():10.3e}, mean={slice_2d.mean():10.3e}")

print("\n" + "="*70)

# =============================================================================
# CELL 3c: Summary Statistics for 2D Slices at z=8
# =============================================================================

print("\n" + "="*70)
print(f"SUMMARY STATISTICS FOR 2D SLICES AT z ≈ {target_z}")
print("="*70)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    
    # Find the closest redshift slice
    z_values = lightcone.lightcone_redshifts
    closest_idx = np.argmin(np.abs(z_values - target_z))
    actual_z = z_values[closest_idx]
    
    # Extract 2D slices
    brightness_slice = lightcone.brightness_temp[:, :, closest_idx]
    xHI_slice = lightcone.xH_box[:, :, closest_idx]
    density_slice = lightcone.density[:, :, closest_idx]  # δ
    velocity_slice = lightcone.velocity[:, :, closest_idx]
    
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    print(f"\nBOX_LEN = {box_len:.0f} Mpc (Cell = {cell_size_mpc:.2f} Mpc = {cell_size_kpc:.0f} kpc) at z = {actual_z:.2f}:")
    print(f"  Brightness temp [mK]: min={brightness_slice.min():.2f}, "
          f"max={brightness_slice.max():.2f}, mean={brightness_slice.mean():.2f}")
    print(f"  Neutral fraction:     min={xHI_slice.min():.4f}, "
          f"max={xHI_slice.max():.4f}, mean={xHI_slice.mean():.4f}")
    print(f"  Overdensity δ:        min={density_slice.min():.3f}, "
          f"max={density_slice.max():.3f}, mean={density_slice.mean():.3f}")
    print(f"  Velocity [km/s]:      min={velocity_slice.min():.2f}, "
          f"max={velocity_slice.max():.2f}, mean={velocity_slice.mean():.2f}")

print("\n" + "="*70)

# =============================================================================
# CELL 3d: Summary Statistics for Full Lightcones (All Box Sizes)
# =============================================================================

print("\n" + "="*70)
print("SUMMARY STATISTICS FOR FULL LIGHTCONES (ALL BOX SIZES)")
print("="*70)
print(f"Fixed: HII_DIM = {HII_DIM_FIXED}")
print(f"Total simulations: {len(BOX_LEN_VALUES)}")

for idx, box_len in enumerate(BOX_LEN_VALUES):
    if box_len not in lightcones or lightcones[box_len] is None:
        print(f"\n[{idx+1}/{len(BOX_LEN_VALUES)}] BOX_LEN = {box_len:.0f} Mpc: FAILED")
        continue
        
    lightcone = lightcones[box_len]
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    print(f"\n[{idx+1}/{len(BOX_LEN_VALUES)}] BOX_LEN = {box_len:.0f} Mpc (Cell = {cell_size_mpc:.2f} Mpc = {cell_size_kpc:.0f} kpc):")
    print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
          f"{lightcone.lightcone_redshifts.max():.2f}]")
    
    # Full 3D field statistics
    print(f"  Brightness temp [mK]: min={lightcone.brightness_temp.min():.2f}, "
          f"max={lightcone.brightness_temp.max():.2f}, "
          f"mean={lightcone.brightness_temp.mean():.2f}")
    print(f"  Neutral fraction:     min={lightcone.xH_box.min():.4f}, "
          f"max={lightcone.xH_box.max():.4f}, "
          f"mean={lightcone.xH_box.mean():.4f}")
    print(f"  Overdensity δ:        min={lightcone.density.min():.3f}, "
          f"max={lightcone.density.max():.3f}, "
          f"mean={lightcone.density.mean():.3f}")
    print(f"  Velocity [km/s]:      min={lightcone.velocity.min():.2f}, "
          f"max={lightcone.velocity.max():.2f}, "
          f"mean={lightcone.velocity.mean():.2f}")

print("\n" + "="*70)

# =============================================================================
# CELL 3e: Compact Summary Table
# =============================================================================

print("\n" + "="*70)
print("COMPACT SUMMARY: REIONIZATION PROGRESS BY BOX SIZE")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'Cell[kpc]':<12} {'<xHI>':<10} {'<Tb>[mK]':<12} {'z_range':<15}")
print("-" * 70)

for box_len in BOX_LEN_VALUES:
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    cell_size_kpc = (box_len / HII_DIM_FIXED) * 1000
    mean_xHI = lightcone.xH_box.mean()
    mean_Tb = lightcone.brightness_temp.mean()
    z_min_actual = lightcone.lightcone_redshifts.min()
    z_max_actual = lightcone.lightcone_redshifts.max()
    
    print(f"{box_len:<12.0f} {cell_size_kpc:<12.0f} {mean_xHI:<10.4f} {mean_Tb:<12.2f} "
          f"[{z_min_actual:.2f}, {z_max_actual:.2f}]")

print("="*70)


GENERATING 2D SLICES AT z = 8.0
Using all 5 BOX_LEN values for slices

Processing brightness_temp...
  ✓ Saved: brightness_temp_slice_z8_boxsize_all
    Range: [0.000, 25.470]

Processing xH_box...
  ✓ Saved: xH_box_slice_z8_boxsize_all
    Range: [0.000, 1.000]

Processing density...
  ✓ Saved: density_slice_z8_boxsize_all
    Range: [-0.536, 1.827]

Processing velocity...
  ✓ Saved: velocity_slice_z8_boxsize_all
    Range: [-0.000, 0.000]

✓ 2D SLICE PLOTTING COMPLETE!

2D SLICE STATISTICS AT z = 8

BOX_LEN = 400 Mpc (Cell = 3.12 Mpc = 3125 kpc) at z = 8.00:
  brightness_temp: min= 0.000e+00, max= 2.547e+01, mean= 1.202e+01
  xH_box         : min= 0.000e+00, max= 1.000e+00, mean= 5.457e-01
  density        : min=-5.358e-01, max= 1.827e+00, mean= 6.200e-03
  velocity       : min=-1.198e-16, max= 1.027e-16, mean= 7.610e-18

BOX_LEN = 800 Mpc (Cell = 6.25 Mpc = 6250 kpc) at z = 7.99:
  brightness_temp: min= 0.000e+00, max= 1.879e+01, mean= 1.126e+01
  xH_box         : min= 0.000e+00, m

In [11]:
# =============================================================================
# CELL 4: Reionization History Analysis - Box Size Scan
# =============================================================================
print("\n" + "="*70)
print("GENERATING REIONIZATION HISTORY COMPARISON")
print("="*70)

# Create rainbow colormap for all box sizes
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

# =============================================================================
# PLOT 4a: Ionization Fraction vs Redshift (All Box Sizes)
# =============================================================================

def plot_reionization_xe(ax):
    for box_len in BOX_LEN_VALUES:
        if box_len not in lightcones or lightcones[box_len] is None:
            continue
            
        lightcone = lightcones[box_len]
        z_nodes = lightcone.node_redshifts[::-1]
        x_e_nodes = 1.0 - lightcone.global_xH[::-1]
        
        color = cmap(norm(box_len))
        
        ax.plot(
            z_nodes,
            x_e_nodes,
            linewidth=2.5,
            color=color,
            marker='o',
            markersize=3,
            alpha=0.8,
            label=f'{box_len:.0f} Mpc'
        )

    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Ionization Fraction $x_e$')
    ax.set_ylim(-0.05, 1.05)
    ax.invert_xaxis()
    ax.legend(loc='best', ncol=1)

plot_name = "reionization_history_xe_boxsize_all"

save_pdf_png(
    plot_reionization_xe,  # ← Pass the function
    plot_dir,
    plot_name,
    title=rf'Reionization History: Ionization Fraction (Fixed Resolution = {CELL_SIZE_MPC:.2f} Mpc)'
)

print(f"✓ Saved: {plot_name}")

# =============================================================================
# CELL 4f: Optical Depth Calculations for All Box Sizes
# =============================================================================

print("\n" + "="*70)
print("OPTICAL DEPTH CALCULATIONS")
print("="*70)

# Physical constants for optical depth calculation
c_km_s = 2.998e5                    # Speed of light [km/s]
h = 0.6766                          # Hubble parameter (from default cosmology)
H0 = 100 * h                        # Hubble constant [km/s/Mpc]
Omega_b = 0.04897468161869667       # Baryon density (from default cosmology)
Omega_m = 0.30964144154550644       # Matter density (from default cosmology)

# Critical density of the universe [protons/cm^3]
rho_crit_p_cm3 = 1.88e-29 * h**2 / (1.67e-24)  # Convert to protons/cm^3
n_H0_cm3 = Omega_b * rho_crit_p_cm3             # Mean hydrogen number density [cm^-3]

# Thomson scattering cross section
sigma_T_cm2 = 6.65e-25              # [cm^2]

# Convert to Mpc units
cm_per_Mpc = 3.086e24
n_e0_Mpc3 = n_H0_cm3 * cm_per_Mpc**3
sigma_T_Mpc2 = sigma_T_cm2 / cm_per_Mpc**2

# Prefactor for dτ calculation
prefactor = n_e0_Mpc3 * sigma_T_Mpc2  # [Mpc^-1]

print(f"\nPhysical constants:")
print(f"  n_H0 = {n_H0_cm3:.6e} cm^-3")
print(f"  σ_T = {sigma_T_cm2:.6e} cm^2")
print(f"  Prefactor = {prefactor:.6e} Mpc^-1")

# Dictionary to store optical depth results
tau_results = {}

# ← CHANGED: Loop through both arrays
for box_len, hii_dim in zip(BOX_LEN_VALUES, HII_DIM_VALUES):
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    lightcone = lightcones[box_len]
    
    # Extract redshift and distance axes
    red_axis = lightcone.lightcone_redshifts
    pos_axis = lightcone.lightcone_distances  # Comoving distance [Mpc]
    
    # Trim to z <= z_max if needed
    ind_z = np.where(red_axis <= z_max)[0]
    red_axis = red_axis[ind_z]
    pos_axis = pos_axis[ind_z]
    
    # Get ionization history
    z_nodes_sorted = lightcone.node_redshifts[::-1]
    xHI_nodes_sorted = lightcone.global_xH[::-1]
    x_e_nodes_sorted = 1.0 - xHI_nodes_sorted
    
    # Interpolate x_e onto lightcone redshift grid
    x_e_interp = np.interp(red_axis, z_nodes_sorted, x_e_nodes_sorted)
    
    # Calculate geometric quantities
    s = pos_axis  # Comoving distance [Mpc]
    ds = np.diff(s)  # Distance element [Mpc]
    
    # Midpoint values for integration
    z_mid = 0.5 * (red_axis[:-1] + red_axis[1:])
    x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])
    
    # Calculate dτ
    dtau = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds
    
    # Cumulative optical depth
    tau = np.cumsum(dtau)
    tau_total = tau[-1]
    
    # Store results (including hii_dim for reference)  # ← CHANGED
    tau_results[box_len] = {
        'red_axis': red_axis,
        'pos_axis': pos_axis,
        's': s,
        'ds': ds,
        'z_mid': z_mid,
        'x_e_interp': x_e_interp,
        'x_e_mid': x_e_mid,
        'dtau': dtau,
        'tau': tau,
        'tau_total': tau_total,
        'hii_dim': hii_dim  # ← CHANGED: Store HII_DIM
    }

# Print compact summary  # ← CHANGED
print(f"\n{'BOX[Mpc]':<12} {'HII_DIM':<10} {'Cell[Mpc]':<12} {'z_range':<15} {'<ds>[Mpc]':<12} {'τ_total':<10}")
print("-" * 85)

for box_len in sorted(tau_results.keys()):
    results = tau_results[box_len]
    red_axis = np.asarray(results['red_axis'])
    ds = np.asarray(results['ds'])
    tau_total = float(np.asarray(results['tau_total']))
    hii_dim = results['hii_dim']  # ← CHANGED
    cell_size_mpc = box_len / hii_dim  # ← CHANGED
    
    print(f"{box_len:<12.0f} {hii_dim:<10} {cell_size_mpc:<12.3f} [{red_axis.min():.2f}, {red_axis.max():.2f}] "
          f"{ds.mean():<12.3f} {tau_total:<10.6f}")

print("\n" + "="*70)

# =============================================================================
# PLOT: Cumulative Optical Depth τ vs z (All Box Sizes - Rainbow)
# =============================================================================

def plot_tau_vs_z(ax):
    for box_len in sorted(tau_results.keys()):
        results = tau_results[box_len]
        
        z_mid_plot = np.asarray(results['z_mid'])
        tau_plot = np.asarray(results['tau'])
        
        color = cmap(norm(box_len))
        
        ax.plot(
            z_mid_plot,
            tau_plot,
            linewidth=2.5,
            color=color,
            marker='o',
            markersize=3,
            alpha=0.8,
            label=f'{box_len:.0f} Mpc'
        )

    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$')
    ax.invert_xaxis()
    ax.legend(loc='best', ncol=1)

plot_name = "tau_vs_z_boxsize_all"

save_pdf_png(
    plot_tau_vs_z,
    plot_dir,
    plot_name,
    title=rf'Cumulative Optical Depth vs Redshift (Fixed Resolution = {CELL_SIZE_MPC:.2f} Mpc)'
)

print(f"✓ Saved: {plot_name}")


# =============================================================================
# PLOT: Optical Depth Element dτ vs z (All Box Sizes - Rainbow)
# =============================================================================

def plot_dtau_vs_z(ax):
    for box_len in sorted(tau_results.keys()):
        results = tau_results[box_len]
        z_mid_plot = np.asarray(results['z_mid'])
        dtau_plot = np.asarray(results['dtau'])
        
        color = cmap(norm(box_len))
        
        ax.plot(
            z_mid_plot,
            dtau_plot,
            linewidth=2.5,
            color=color,
            marker='o',
            markersize=3,
            alpha=0.8,
            label=f'{box_len:.0f} Mpc'
        )

    ax.set_xlabel(r'Redshift $z$')
    ax.set_ylabel(r'Optical Depth Element $d\tau$')
    ax.invert_xaxis()
    ax.legend(loc='best', ncol=1)

plot_name = "dtau_vs_z_boxsize_all"

save_pdf_png(
    plot_dtau_vs_z,
    plot_dir,
    plot_name,
    title=r'Optical Depth Element $d\tau$ vs Redshift'
)

print(f"✓ Saved: {plot_name}")


# =============================================================================
# PLOT: Total Optical Depth vs BOX_LEN
# =============================================================================

# Prepare data outside function
box_vals = sorted(tau_results.keys())
tau_total_vals = [float(np.asarray(tau_results[b]['tau_total'])) for b in box_vals]
cell_sizes_mpc = [box_vals[i] / tau_results[box_vals[i]]['hii_dim'] for i in range(len(box_vals))]

def plot_tau_total_vs_boxsize(ax):
    ax.plot(
        box_vals,
        tau_total_vals,
        'o-',
        linewidth=3,
        markersize=10,
        color='darkblue',
        label=r'Total $\tau$'
    )

    ax.set_xlabel(r'BOX\_LEN [Mpc]')
    ax.set_ylabel(r'Total Optical Depth $\tau$')
    ax.legend(loc='best')

    ax.text(
        0.05, 0.95,
        rf'Fixed Resolution = {CELL_SIZE_MPC:.2f} Mpc',
        transform=ax.transAxes,
        fontsize=14,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    )

plot_name = "tau_total_vs_boxsize"

save_pdf_png(
    plot_tau_total_vs_boxsize,
    plot_dir,
    plot_name,
    title="Total Optical Depth vs Box Size"
)

print(f"✓ Saved: {plot_name}")


# =============================================================================
# PLOT: Total Optical Depth vs Cell Size
# =============================================================================

def plot_tau_total_vs_cellsize(ax):
    ax.plot(
        cell_sizes_mpc,
        tau_total_vals,
        's-',
        linewidth=3,
        markersize=10,
        color='darkred',
        label=r'Total $\tau$'
    )

    ax.set_xlabel(r'Cell Size [Mpc]')
    ax.set_ylabel(r'Total Optical Depth $\tau$')
    ax.legend(loc='best')

    ax.text(
        0.05, 0.95,
        rf'Fixed Resolution = {CELL_SIZE_MPC:.2f} Mpc',
        transform=ax.transAxes,
        fontsize=14,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    )

plot_name = "tau_total_vs_cellsize"

save_pdf_png(
    plot_tau_total_vs_cellsize,
    plot_dir,
    plot_name,
    title="Total Optical Depth vs Cell Size (Should be constant)"
)

print(f"✓ Saved: {plot_name}")


GENERATING REIONIZATION HISTORY COMPARISON
✓ Saved: reionization_history_xe_boxsize_all

OPTICAL DEPTH CALCULATIONS

Physical constants:
  n_H0 = 2.523928e-07 cm^-3
  σ_T = 6.650000e-25 cm^2
  Prefactor = 5.179580e-07 Mpc^-1

BOX[Mpc]     HII_DIM    Cell[Mpc]    z_range         <ds>[Mpc]    τ_total   
-------------------------------------------------------------------------------------
200          64         3.125        [5.00, 19.99] 3.128        0.037267  
600          192        3.125        [5.00, 19.99] 3.128        0.037043  
1000         320        3.125        [5.00, 19.99] 3.128        0.037025  
1400         448        3.125        [5.00, 19.99] 3.128        0.037026  

✓ Saved: tau_vs_z_boxsize_all
✓ Saved: dtau_vs_z_boxsize_all
✓ Saved: tau_total_vs_boxsize
✓ Saved: tau_total_vs_cellsize


In [12]:
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function for All Box Sizes
# kSZ integrand = (1 + δ) × x_e × v_z / c × e^(-τ(z))
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION")
print("="*70)

# Speed of light in km/s
c_km_s = 299792.458  # km/s

print(f"Speed of light: c = {c_km_s:.6e} km/s")

# Dictionary to store kSZ results
kSZ_results = {}

# ← CHANGED: Loop through both arrays
for box_len, hii_dim in zip(BOX_LEN_VALUES, HII_DIM_VALUES):
    if box_len not in lightcones or lightcones[box_len] is None:
        continue
        
    if box_len not in tau_results:
        continue
    
    cell_size_mpc = box_len / hii_dim  # ← CHANGED
    
    print(f"\n{'='*70}")
    print(f"BOX_LEN = {box_len:.0f} Mpc, HII_DIM = {hii_dim} (Cell = {cell_size_mpc:.3f} Mpc)")  # ← CHANGED
    print(f"{'='*70}")
    
    lightcone = lightcones[box_len]
    results = tau_results[box_len]
    
    # Extract redshift and distance axes (strip units)
    red_axis = np.asarray(results['red_axis'])
    z_mid = np.asarray(results['z_mid'])
    tau = np.asarray(results['tau'])
    
    # Extract 3D fields
    red_axis_full = np.asarray(lightcone.lightcone_redshifts)
    ind_z = np.where(red_axis_full <= z_max)[0]
    
    density_1plus = 1 + np.asarray(lightcone.density[:, :, ind_z])  # 1 + δ
    x_e_3D = 1 - np.asarray(lightcone.xH_box[:, :, ind_z])          # Ionized fraction
    v_los_km_s = np.asarray(lightcone.velocity[:, :, ind_z])*3.085677581e19  # Velocity [km/s]
    
    print(f"3D field shapes: {density_1plus.shape}")
    
    # =============================================================================
    # Interpolate τ(z) onto lightcone redshifts
    # =============================================================================
    
    tau_extended = np.concatenate([[0], tau])
    tau_at_lightcone = np.interp(red_axis, 
                                  np.concatenate([[red_axis[0]], z_mid]), 
                                  tau_extended)
    
    print(f"τ range: [{tau_at_lightcone.min():.6f}, {tau_at_lightcone.max():.6f}]")
    
    # Visibility function e^(-τ)
    visibility = np.exp(-tau_at_lightcone)
    print(f"e^(-τ) range: [{visibility.min():.6f}, {visibility.max():.6f}]")
    
    # Broadcast visibility to 3D
    visibility_3D = visibility[None, None, :]  # Shape (1, 1, n_redshift)
    
    # =============================================================================
    # Compute kSZ integrand WITH visibility function
    # =============================================================================
    
    kSZ_integrand = density_1plus * x_e_3D * v_los_km_s / c_km_s * visibility_3D
    
    print(f"\nkSZ INTEGRAND (with visibility) STATISTICS:")
    print(f"  Mean: {kSZ_integrand.mean():.4e}")
    print(f"  Std:  {kSZ_integrand.std():.4e}")
    print(f"  Min:  {kSZ_integrand.min():.4e}")
    print(f"  Max:  {kSZ_integrand.max():.4e}")
    print(f"  RMS:  {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")
    
    # Store results (including hii_dim)  # ← CHANGED
    kSZ_results[box_len] = {
        'kSZ_integrand': kSZ_integrand,
        'visibility': visibility,
        'visibility_3D': visibility_3D,
        'tau_at_lightcone': tau_at_lightcone,
        'red_axis': red_axis,
        'ind_z': ind_z,
        'hii_dim': hii_dim  # ← CHANGED: Store HII_DIM
    }
    
    # Add to lightcone object
    lightcone.kSZ_integrand = kSZ_integrand
    lightcone.visibility_func = visibility_3D

print("\n" + "="*70)
print("kSZ INTEGRAND CALCULATION COMPLETE")
print(f"Computed for {len(kSZ_results)} BOX_LEN values")
print("="*70)


# =============================================================================
# Summary Statistics Table
# =============================================================================

print("\n" + "="*70)
print("kSZ INTEGRAND SUMMARY STATISTICS")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'HII_DIM':<10} {'Cell[Mpc]':<12} {'Mean':<12} {'Std':<12} {'RMS':<12} {'Max|val|':<12}")  # ← CHANGED
print("-" * 95)  # ← CHANGED: wider line

for box_len in sorted(kSZ_results.keys()):
    results = kSZ_results[box_len]
    kSZ_int = results['kSZ_integrand']
    hii_dim = results['hii_dim']  # ← CHANGED
    
    cell_size_mpc = box_len / hii_dim  # ← CHANGED
    mean_val = kSZ_int.mean()
    std_val = kSZ_int.std()
    rms_val = np.sqrt(np.mean(kSZ_int**2))
    max_abs_val = np.max(np.abs(kSZ_int))
    
    print(f"{box_len:<12.0f} {hii_dim:<10} {cell_size_mpc:<12.3f} {mean_val:<12.4e} {std_val:<12.4e} "  # ← CHANGED
          f"{rms_val:<12.4e} {max_abs_val:<12.4e}")


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION
Speed of light: c = 2.997925e+05 km/s

BOX_LEN = 200 Mpc, HII_DIM = 64 (Cell = 3.125 Mpc)
3D field shapes: (64, 64, 963)
τ range: [0.000000, 0.037267]
e^(-τ) range: [0.963419, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 3.7234e-06
  Std:  1.8310e-03
  Min:  -3.0732e-02
  Max:  2.7663e-02
  RMS:  1.8310e-03

BOX_LEN = 600 Mpc, HII_DIM = 192 (Cell = 3.125 Mpc)
3D field shapes: (192, 192, 963)
τ range: [0.000000, 0.037043]
e^(-τ) range: [0.963635, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 9.9334e-06
  Std:  2.2595e-03
  Min:  -5.1222e-02
  Max:  4.4159e-02
  RMS:  2.2595e-03

BOX_LEN = 1000 Mpc, HII_DIM = 320 (Cell = 3.125 Mpc)
3D field shapes: (320, 320, 963)
τ range: [0.000000, 0.037025]
e^(-τ) range: [0.963652, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 1.3269e-05
  Std:  2.3623e-03
  Min:  -6.9264e-02
  Max:  4.9447e-02
  RMS:  2.3624e-03

BOX_LEN = 1400 Mpc, HII_DIM = 448 (Ce

In [ ]:
# =============================================================================
# NOT FOR REPORTS
# CEll5b: PLOT: kSZ Integrand - All Box Sizes Stacked
# =============================================================================

print("\n" + "="*70)
print("GENERATING kSZ INTEGRAND PLOTS")
print("="*70)

# Stacked plots for all box sizes
fig, axes = plt.subplots(len(BOX_LEN_VALUES), 1, 
                         figsize=(14, 4*len(BOX_LEN_VALUES)), 
                         constrained_layout=True)

if len(BOX_LEN_VALUES) == 1:
    axes = [axes]

# Rainbow colormap for labels
cmap = mpl.cm.rainbowprint("v_los_km_s stats [km/s]:",
      np.min(v_los_km_s),
      np.mean(v_los_km_s),
      np.max(v_los_km_s),
      np.std(v_los_km_s))
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
    if box_len not in kSZ_results:
        ax.text(0.5, 0.5, f'BOX_LEN = {box_len:.0f} Mpc\nNo data', 
               ha='center', va='center',
               transform=ax.transAxes, fontsize=16, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        continue
        
    lightcone = lightcones[box_len]
    
    plotting.lightcone_sliceplot(lightcone, 'kSZ_integrand', ax=ax, fig=fig)
    
    # Change colormap to seismic (diverging colormap for positive/negative)
    im = ax.images[0]
    im.set_cmap('seismic')
    
    # Set symmetric color limits
    kSZ_data = kSZ_results[box_len]['kSZ_integrand']
    vmax = np.percentile(np.abs(kSZ_data), 99)
    im.set_clim(-vmax, vmax)
    
    # Get rainbow color for label
    color = cmap(norm(box_len))
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    # Add label with box info
    ax.text(0.02, 0.98, 
           f'BOX = {box_len:.0f} Mpc  |  Cell = {cell_size_mpc:.2f} Mpc ({cell_size_kpc:.0f} kpc)', 
           transform=ax.transAxes, 
           fontsize=13, fontweight='bold',
           verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor=color, alpha=0.7, 
                    edgecolor='black', linewidth=2))
    
    # Add box size on right
    ax.text(0.98, 0.98, 
           f'{box_len:.0f} Mpc', 
           transform=ax.transAxes, fontsize=14, fontweight='bold',
           verticalalignment='top', horizontalalignment='right',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Save
plot_name = "kSZ_integrand_with_visibility_boxsize_stack"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

fig.suptitle(r'kSZ Integrand: $(1+\delta) \times x_e \times v_z/c \times e^{-\tau(z)}$ (HII_DIM=' + f'{HII_DIM_FIXED})', 
             fontsize=20, fontweight='bold', y=0.995)
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Visibility Function e^(-τ) vs z (All Box Sizes - Rainbow)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(kSZ_results.keys()):
    results = kSZ_results[box_len]
    red_axis_plot = results['red_axis']
    visibility_plot = results['visibility']
    
    color = cmap(norm(box_len))
    
    ax.plot(red_axis_plot, visibility_plot, 
           linewidth=2.5, color=color,
           marker='o', markersize=3, alpha=0.8,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Visibility Function $e^{-\tau(z)}$', fontsize=20)
ax.set_ylim(0, 1.05)
ax.invert_xaxis()
ax.legend(fontsize=12, loc='best', ncol=1)
##ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

plot_name = "visibility_function_vs_z_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Visibility Function $e^{-\tau(z)}$ vs Redshift', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n" + "="*70)
print("kSZ INTEGRAND PLOTTING COMPLETE!")
print("="*70)

In [13]:
# =============================================================================
# CELL 6: Compute Line-of-Sight Integrated kSZ Maps for All Box Sizes
# kSZ(z=5) = ∫ n_e0 σ_T (1/a²) (1+δ) x_e v_z/c e^(-τ) ds
# Integration from z=20 to z=5 along line of sight
# =============================================================================

print("\n" + "="*70)
print("LINE-OF-SIGHT kSZ MAP INTEGRATION")
print("="*70)

# Physical constants in CGS
print(f"\n=== PHYSICAL CONSTANTS (CGS) ===")
c_cm_s = 3.0e10  # cm/s
sigma_T_cm2 = 6.6525e-25  # cm²
n_e0_cm3 = 2.06e-7  # cm⁻³
Mpc_to_cm = 3.0857e24  # cm/Mpc

print(f"c = {c_cm_s:.2e} cm/s")
print(f"σ_T = {sigma_T_cm2:.4e} cm²")
print(f"n_e0 = {n_e0_cm3:.4e} cm⁻³")
print(f"1 Mpc = {Mpc_to_cm:.4e} cm")

# Calculate dimensionless prefactor
prefactor_cgs = n_e0_cm3 * sigma_T_cm2 * c_cm_s  # [1/s]
print(f"\nPrefactor n_e0 × σ_T × c = {prefactor_cgs:.4e} s⁻¹")

# Dictionary to store kSZ map results
kSZ_map_results = {}

# ← CHANGED: Loop through both arrays
for box_len, hii_dim in zip(BOX_LEN_VALUES, HII_DIM_VALUES):
    if box_len not in kSZ_results or box_len not in tau_results:
        continue
    
    cell_size_mpc = box_len / hii_dim  # ← CHANGED
    
    print(f"\n{'='*70}")
    print(f"BOX_LEN = {box_len:.0f} Mpc, HII_DIM = {hii_dim} (Cell = {cell_size_mpc:.3f} Mpc)")  # ← CHANGED
    print(f"{'='*70}")
    
    lightcone = lightcones[box_len]
    results = kSZ_results[box_len]
    
    # Extract arrays
    red_axis = np.asarray(results['red_axis'])
    kSZ_integrand_with_vis = results['kSZ_integrand']
    
    # Get ds from tau_results
    ds_raw = tau_results[box_len]['ds']
    ds_Mpc = np.asarray(ds_raw)
    
    print(f"ds range: {ds_Mpc.min():.6f} - {ds_Mpc.max():.6f} Mpc")
    
    # Convert ds from Mpc to cm
    ds_cm = ds_Mpc * Mpc_to_cm  # cm
    print(f"ds in cm: {ds_cm.min():.4e} - {ds_cm.max():.4e} cm")
    
    # =============================================================================
    # Prepare integrand
    # =============================================================================
    
    # Scale factor a = 1/(1+z)
    a = 1.0 / (1.0 + red_axis)
    a_squared = a**2
    
    # Midpoint kSZ integrand
    kSZ_integrand_mid = 0.5 * (kSZ_integrand_with_vis[:, :, :-1] + 
                                kSZ_integrand_with_vis[:, :, 1:])
    
    # Midpoint a²
    a_squared_mid = 0.5 * (a_squared[:-1] + a_squared[1:])
    a_squared_mid_3D = a_squared_mid[None, None, :]
    
    # Full dimensionless integrand
    kSZ_integrand_full = (prefactor_cgs / a_squared_mid_3D) * kSZ_integrand_mid * (ds_cm / c_cm_s)[None, None, :]
    
    print(f"\nIntegrand shape: {kSZ_integrand_full.shape}")
    print(f"Integrating over {kSZ_integrand_full.shape[2]} redshift slices")
    print(f"From z = {red_axis.max():.2f} to z = {red_axis.min():.2f}")
    
    # =============================================================================
    # Integrate along line of sight
    # =============================================================================
    
    kSZ_map = np.sum(kSZ_integrand_full, axis=2)  # Shape (HII_DIM, HII_DIM), dimensionless
    
    print(f"\n=== kSZ MAP STATISTICS (DIMENSIONLESS) ===")
    print(f"Shape: {kSZ_map.shape}")
    print(f"Mean: {kSZ_map.mean():.4e}")
    print(f"Std:  {kSZ_map.std():.4e}")
    print(f"Min:  {kSZ_map.min():.4e}")
    print(f"Max:  {kSZ_map.max():.4e}")
    print(f"RMS:  {np.sqrt(np.mean(kSZ_map**2)):.4e}")
    
    # Store results (including hii_dim)  # ← CHANGED
    kSZ_map_results[box_len] = {
        'kSZ_map': kSZ_map,
        'kSZ_integrand_full': kSZ_integrand_full,
        'hii_dim': hii_dim  # ← CHANGED: Store HII_DIM
    }
    
    # Add to lightcone object
    lightcone.kSZ_map = kSZ_map

print("\n" + "="*70)
print("kSZ MAP INTEGRATION COMPLETE")
print(f"Computed maps for {len(kSZ_map_results)} BOX_LEN values")
print("="*70)

# =============================================================================
# Summary Statistics Table
# =============================================================================

print("\n" + "="*70)
print("kSZ MAP SUMMARY STATISTICS")
print("="*70)
print(f"{'BOX[Mpc]':<12} {'HII_DIM':<10} {'Cell[Mpc]':<12} {'Mean':<12} {'Std':<12} {'RMS':<12} {'Max|val|':<12}")  # ← CHANGED
print("-" * 95)  # ← CHANGED: wider line

for box_len in sorted(kSZ_map_results.keys()):
    results = kSZ_map_results[box_len]  # ← CHANGED: get full results dict
    kSZ_map = results['kSZ_map']
    hii_dim = results['hii_dim']  # ← CHANGED
    
    cell_size_mpc = box_len / hii_dim  # ← CHANGED
    mean_val = kSZ_map.mean()
    std_val = kSZ_map.std()
    rms_val = np.sqrt(np.mean(kSZ_map**2))
    max_abs_val = np.max(np.abs(kSZ_map))
    
    print(f"{box_len:<12.0f} {hii_dim:<10} {cell_size_mpc:<12.3f} {mean_val:<12.4e} {std_val:<12.4e} "  # ← CHANGED
          f"{rms_val:<12.4e} {max_abs_val:<12.4e}")


LINE-OF-SIGHT kSZ MAP INTEGRATION

=== PHYSICAL CONSTANTS (CGS) ===
c = 3.00e+10 cm/s
σ_T = 6.6525e-25 cm²
n_e0 = 2.0600e-07 cm⁻³
1 Mpc = 3.0857e+24 cm

Prefactor n_e0 × σ_T × c = 4.1112e-21 s⁻¹

BOX_LEN = 200 Mpc, HII_DIM = 64 (Cell = 3.125 Mpc)
ds range: 3.128242 - 3.128242 Mpc
ds in cm: 9.6528e+24 - 9.6528e+24 cm

Integrand shape: (64, 64, 962)
Integrating over 962 redshift slices
From z = 19.99 to z = 5.00

=== kSZ MAP STATISTICS (DIMENSIONLESS) ===
Shape: (64, 64)
Mean: 2.7706e-07
Std:  1.5109e-05
Min:  -9.8504e-05
Max:  8.2893e-05
RMS:  1.5112e-05

BOX_LEN = 600 Mpc, HII_DIM = 192 (Cell = 3.125 Mpc)
ds range: 3.128242 - 3.128242 Mpc
ds in cm: 9.6528e+24 - 9.6528e+24 cm

Integrand shape: (192, 192, 962)
Integrating over 962 redshift slices
From z = 19.99 to z = 5.00

=== kSZ MAP STATISTICS (DIMENSIONLESS) ===
Shape: (192, 192)
Mean: 4.6035e-07
Std:  1.3109e-05
Min:  -8.2177e-05
Max:  6.3353e-05
RMS:  1.3117e-05

BOX_LEN = 1000 Mpc, HII_DIM = 320 (Cell = 3.125 Mpc)
ds range: 3.128

In [ ]:
# =============================================================================
# Not FOR REPORTs
# Cell 6b:  PLOT: kSZ Maps - All Box Sizes Side-by-Side
# =============================================================================

print("\n" + "="*70)
print("GENERATING kSZ MAP PLOTS")
print("="*70)

fig, axes = plt.subplots(1, len(BOX_LEN_VALUES), 
                         figsize=(5*len(BOX_LEN_VALUES), 5.5), 
                         constrained_layout=True)

if len(BOX_LEN_VALUES) == 1:
    axes = [axes]

# Rainbow colormap for labels
cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(vmin=BOX_LEN_VALUES.min(), vmax=BOX_LEN_VALUES.max())

# Find global vmax for consistent color scale
vmax_global = 0
for box_len in kSZ_map_results.keys():
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    vmax_global = max(vmax_global, np.percentile(np.abs(kSZ_map), 99))

for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
    if box_len not in kSZ_map_results:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
               transform=ax.transAxes, fontsize=16)
        ax.set_title(f'BOX={box_len:.0f} Mpc', fontsize=14)
        continue
        
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    # Plot the kSZ map
    im = ax.imshow(kSZ_map.T,
                   cmap='seismic',
                   origin='lower',
                   extent=[0, box_len, 0, box_len],
                   aspect='equal',
                   vmin=-vmax_global,
                   vmax=vmax_global)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)
    
    # Labels
    ax.set_xlabel('x [Mpc]', fontsize=11)
    ax.set_ylabel('y [Mpc]', fontsize=11)
    
    # Get rainbow color for title
    color_label = cmap(norm(box_len))
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    ax.set_title(f'BOX={box_len:.0f} Mpc\nCell={cell_size_mpc:.2f} Mpc\n({cell_size_kpc:.0f} kpc)', 
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor=color_label, alpha=0.6, 
                         edgecolor='black', linewidth=1.5))

# Overall title
fig.suptitle(r'kSZ Maps at $z=5$ (line-of-sight integrated, HII_DIM=' + f'{HII_DIM_FIXED})', 
             fontsize=18, fontweight='bold')

# Save
plot_name = "kSZ_maps_z5_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: kSZ Maps - Stacked Vertically
# =============================================================================

fig, axes = plt.subplots(len(BOX_LEN_VALUES), 1, 
                         figsize=(8, 6*len(BOX_LEN_VALUES)), 
                         constrained_layout=True)

if len(BOX_LEN_VALUES) == 1:
    axes = [axes]

for idx, (box_len, ax) in enumerate(zip(BOX_LEN_VALUES, axes)):
    if box_len not in kSZ_map_results:
        ax.text(0.5, 0.5, f'BOX={box_len:.0f} Mpc\nNo data', 
               ha='center', va='center',
               transform=ax.transAxes, fontsize=16, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        continue
        
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    # Plot the kSZ map
    im = ax.imshow(kSZ_map.T,
                   cmap='seismic',
                   origin='lower',
                   extent=[0, box_len, 0, box_len],
                   aspect='equal',
                   vmin=-vmax_global,
                   vmax=vmax_global)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label('kSZ (dimensionless)', fontsize=11)
    
    # Labels
    ax.set_xlabel('x [Mpc]', fontsize=12)
    ax.set_ylabel('y [Mpc]', fontsize=12)
    
    # Get rainbow color for label
    color_label = cmap(norm(box_len))
    cell_size_mpc = box_len / HII_DIM_FIXED
    cell_size_kpc = cell_size_mpc * 1000
    
    # Add label
    ax.text(0.02, 0.98, 
           f'BOX = {box_len:.0f} Mpc  |  Cell = {cell_size_mpc:.2f} Mpc ({cell_size_kpc:.0f} kpc)', 
           transform=ax.transAxes, fontsize=12, fontweight='bold',
           verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor=color_label, alpha=0.7, 
                    edgecolor='black', linewidth=2))

# Overall title
fig.suptitle(r'kSZ Maps at $z=5$ - Box Size Comparison', 
             fontsize=20, fontweight='bold', y=0.995)

# Save
plot_name = "kSZ_maps_z5_boxsize_stack"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: kSZ Map Histograms - All Box Sizes Overlay
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for box_len in sorted(kSZ_map_results.keys()):
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    color = cmap(norm(box_len))
    
    ax.hist(kSZ_map.flatten(), bins=100, 
           color=color, 
           alpha=0.5, 
           edgecolor='black',
           linewidth=0.5,
           label=f'{box_len:.0f} Mpc')

ax.set_xlabel('kSZ Signal (dimensionless)', fontsize=20)
ax.set_ylabel('Number of Pixels', fontsize=20)
ax.axvline(0, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax.legend(fontsize=14, loc='best')
##ax.grid(True, alpha=0.3, linestyle='--')

# Add colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label(r'BOX\_LEN [Mpc]', fontsize=16)

# Save
plot_name = "kSZ_map_histogram_boxsize_all"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(f'kSZ Map Pixel Distribution (HII_DIM={HII_DIM_FIXED})', 
            fontsize=20, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: kSZ RMS vs Box Size
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

box_vals = sorted(kSZ_map_results.keys())
rms_vals = []
std_vals = []
cell_sizes_kpc = []

for box_len in box_vals:
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    rms_vals.append(np.sqrt(np.mean(kSZ_map**2)))
    std_vals.append(np.std(kSZ_map))
    cell_sizes_kpc.append((box_len / HII_DIM_FIXED) * 1000)

ax.plot(box_vals, rms_vals, 'o-', linewidth=3, markersize=10,
       color='darkblue', label='RMS')
ax.plot(box_vals, std_vals, 's-', linewidth=3, markersize=10,
       color='darkred', label='Std Dev')

ax.set_xlabel(r'BOX\_LEN [Mpc]', fontsize=20)
ax.set_ylabel('kSZ Signal (dimensionless)', fontsize=20)
ax.legend(loc='best', fontsize=16)
##ax.grid(True, alpha=0.3, linestyle='--')

# Add annotation
ax.text(0.05, 0.95, f'HII_DIM = {HII_DIM_FIXED}',
       transform=ax.transAxes, fontsize=14,
       verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plot_name = "kSZ_rms_vs_boxsize"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title('kSZ Map RMS vs Box Size', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)


print("\n" + "="*70)
print("kSZ MAP GENERATION COMPLETE!")
print("="*70)

In [14]:
# =============================================================================
# CELL 7: Compute kSZ Power Spectrum - P(k), C_ℓ, and D_ℓ for All Box Sizes
# WITH ERROR BUDGET (Sample + Cosmic Variance)
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ POWER SPECTRA WITH ERROR BUDGET")
print("="*70)

# CMB temperature conversion
T_CMB_0_K = 2.725  # K
z_obs = 5.0
T_CMB_z5_K = T_CMB_0_K #* (1 + z_obs)  # K
T_CMB_z5_uK = T_CMB_z5_K * 1e6  # μK

print(f"\n=== CMB TEMPERATURE ===")
print(f"T_CMB(z=0) = {T_CMB_0_K:.3f} K")
print(f"T_CMB(z=5) = {T_CMB_z5_K:.3f} K = {T_CMB_z5_uK:.2f} μK")

# Angular diameter distance at z=5
D_A_Mpc = 1300  # Mpc (comoving)
chi_comoving_Mpc = D_A_Mpc * (1 + z_obs)  # = 1300 × 6 = 7800 Mpc

print(f"Angular diameter distance: D_A = {D_A_Mpc:.1f} Mpc")
print(f"Comoving distance to z=5: χ = {chi_comoving_Mpc:.1f} Mpc")

# Dictionary to store power spectrum results
power_spectrum_results = {}

# ← Loop through both arrays
for box_len, hii_dim in zip(BOX_LEN_VALUES, HII_DIM_VALUES):
    if box_len not in kSZ_map_results:
        continue
    
    cell_size_mpc = box_len / hii_dim
    
    print(f"\n{'='*70}")
    print(f"BOX_LEN = {box_len:.0f} Mpc, HII_DIM = {hii_dim} (Cell = {cell_size_mpc:.3f} Mpc)")
    print(f"{'='*70}")
    
    kSZ_map = kSZ_map_results[box_len]['kSZ_map']
    
    # =============================================================================
    # 1. Map Properties
    # =============================================================================
    
    npix_side = kSZ_map.shape[0]
    box_size_Mpc = box_len
    pix_size_Mpc = box_size_Mpc / npix_side
    
    print(f"\n=== MAP PROPERTIES ===")
    print(f"Map size: {npix_side} × {npix_side} pixels")
    print(f"Physical size: {box_size_Mpc:.1f} × {box_size_Mpc:.1f} Mpc²")
    print(f"Pixel size: {pix_size_Mpc:.3f} Mpc/pixel")
    print(f"Map RMS: {np.std(kSZ_map):.4e}")
    
    # Remove mean
    kSZ_map_centered = kSZ_map - np.mean(kSZ_map)
    print(f"Mean subtracted: {np.mean(kSZ_map_centered):.4e}")
    
    # =============================================================================
    # 2. Compute 2D Power Spectrum P(k)
    # =============================================================================
    
    # FFT and shift to center
    fft_map = np.fft.fft2(kSZ_map_centered)
    fft_map_shifted = np.fft.fftshift(fft_map)
    
    # Pixel area in physical units
    pix_area = pix_size_Mpc**2  # Mpc²
    
    # 2D power spectrum: P(k) in [Mpc²]
    ps2d = np.abs(fft_map_shifted)**2 * pix_area / (npix_side**4)
    
    print(f"\n=== 2D POWER SPECTRUM ===")
    print(f"P(k) range: [{ps2d.min():.4e}, {ps2d.max():.4e}] Mpc²")
    
    # =============================================================================
    # 3. k-space grid
    # =============================================================================
    
    # Fundamental frequency
    dk = 2 * np.pi / (npix_side * pix_size_Mpc)  # Mpc⁻¹
    
    # k-space coordinates
    kx = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    ky = np.fft.fftshift(np.fft.fftfreq(npix_side)) * npix_side * dk
    kgrid = np.sqrt(kx[:, None]**2 + ky[None, :]**2)
    
    print(f"\n=== k-SPACE GRID ===")
    print(f"dk (fundamental): {dk:.6f} Mpc⁻¹")
    print(f"k range: [{kgrid.min():.6f}, {kgrid.max():.6f}] Mpc⁻¹")
    
    # =============================================================================
    # 4. Azimuthally Averaged P(k) WITH ERROR BUDGET
    # =============================================================================
    
    # Define k bins (logarithmic spacing)
    k_bins = np.logspace(np.log10(dk), np.log10(kgrid.max()*0.9), 35)
    k_centers = 0.5 * (k_bins[:-1] + k_bins[1:])
    P1d = np.zeros(len(k_centers))
    P1d_err_sample = np.zeros(len(k_centers))  # Sample variance
    n_modes = np.zeros(len(k_centers))  # Number of modes per bin
    
    for i in range(len(k_centers)):
        mask = (kgrid >= k_bins[i]) & (kgrid < k_bins[i+1])
        n_modes[i] = np.sum(mask)
        
        if n_modes[i] > 0:
            values = ps2d[mask]
            P1d[i] = np.mean(values)
            P1d_err_sample[i] = np.std(values) / np.sqrt(n_modes[i])
        else:
            P1d[i] = np.nan
            P1d_err_sample[i] = np.nan
    
    # =============================================================================
    # 4b. COSMIC VARIANCE ERROR
    # =============================================================================
    
    # Number of independent modes in k-space
    # For 3D: V_k = 4π k² dk, but we have a 2D slice
    # Effective k-volume for 2D projection
    k_volume = (box_size_Mpc / (2*np.pi))**2  # For 2D: (L/2π)²
    
    # Number of modes (2D annulus)
    n_modes_cosmic = 2 * np.pi * k_centers * k_volume * (k_bins[1:] - k_bins[:-1])
    
    # Cosmic variance fractional error: 1/√N_modes
    cosmic_var_frac = 1.0 / np.sqrt(n_modes_cosmic)
    cosmic_var_frac[n_modes_cosmic == 0] = np.nan
    
    # Cosmic variance absolute error
    P1d_err_cosmic = P1d * cosmic_var_frac
    
    # =============================================================================
    # 4c. COMBINED ERROR (Sample + Cosmic in quadrature)
    # =============================================================================
    
    P1d_err_total = np.sqrt(P1d_err_sample**2 + P1d_err_cosmic**2)
    
    print(f"\n=== AZIMUTHALLY AVERAGED P(k) WITH ERRORS ===")
    valid_bins = np.sum(~np.isnan(P1d) & (P1d > 0))
    print(f"Valid k bins: {valid_bins} out of {len(k_centers)}")
    print(f"P(k) range: [{np.nanmin(P1d[P1d>0]):.4e}, {np.nanmax(P1d):.4e}] Mpc²")
    
    print(f"\n=== ERROR BUDGET ===")
    valid = ~np.isnan(P1d) & (P1d > 0)
    if np.any(valid):
        rel_err_sample = P1d_err_sample[valid] / P1d[valid]
        rel_err_cosmic = P1d_err_cosmic[valid] / P1d[valid]
        rel_err_total = P1d_err_total[valid] / P1d[valid]
        
        print(f"Sample variance:  {np.nanmean(rel_err_sample)*100:.1f}% (mean)")
        print(f"Cosmic variance:  {np.nanmean(rel_err_cosmic)*100:.1f}% (mean)")
        print(f"Total variance:   {np.nanmean(rel_err_total)*100:.1f}% (mean)")
        print(f"N_modes range:    {n_modes_cosmic[valid].min():.1f} - {n_modes_cosmic[valid].max():.1f}")
    
    # =============================================================================
    # 5. Convert to ℓ space WITH ERRORS
    # =============================================================================
    
    # Convert k to ℓ
    ell_from_k = k_centers * chi_comoving_Mpc/0.67
    
    # C_ℓ from P(k): C_ℓ = P(k) / D_A²
    Cl_approx = P1d*0.67**2 / D_A_Mpc**2
    
    # D_ℓ = ℓ(ℓ+1) C_ℓ / 2π (dimensionless)
    Dl_dimensionless = ell_from_k * (ell_from_k + 1) * Cl_approx / (2 * np.pi)
    
    # D_ℓ in μK²
    Dl_uK2 = Dl_dimensionless * T_CMB_z5_uK**2
    
    # Propagate errors to D_ℓ
    Cl_err_total = P1d_err_total * 0.67**2 / D_A_Mpc**2
    Dl_dimensionless_err = ell_from_k * (ell_from_k + 1) * Cl_err_total / (2 * np.pi)
    Dl_uK2_err_total = Dl_dimensionless_err * T_CMB_z5_uK**2
    
    # Also propagate sample and cosmic separately for plotting
    Cl_err_sample = P1d_err_sample * 0.67**2 / D_A_Mpc**2
    Dl_uK2_err_sample = ell_from_k * (ell_from_k + 1) * Cl_err_sample / (2 * np.pi) * T_CMB_z5_uK**2
    
    Cl_err_cosmic = P1d_err_cosmic * 0.67**2 / D_A_Mpc**2
    Dl_uK2_err_cosmic = ell_from_k * (ell_from_k + 1) * Cl_err_cosmic / (2 * np.pi) * T_CMB_z5_uK**2
    
    print(f"\n=== D_ℓ CONVERSION ===")
    print(f"Conversion factor: T_CMB²(z=5) = {T_CMB_z5_uK**2:.4e} μK²")
    valid_dl = ~np.isnan(Dl_uK2) & (Dl_uK2 > 0)
    if np.any(valid_dl):
        print(f"D_ℓ range: [{np.nanmin(Dl_uK2[valid_dl]):.4e}, {np.nanmax(Dl_uK2[valid_dl]):.4e}] μK²")
        
        rel_err_Dl_total = Dl_uK2_err_total[valid_dl] / Dl_uK2[valid_dl]
        print(f"D_ℓ total error: {np.nanmean(rel_err_Dl_total)*100:.1f}% (mean)")
    
    # Store results (including errors and hii_dim)
    power_spectrum_results[box_len] = {
        'k_centers': k_centers,
        'k_bins': k_bins,
        'P1d': P1d,
        'P1d_err_sample': P1d_err_sample,
        'P1d_err_cosmic': P1d_err_cosmic,
        'P1d_err_total': P1d_err_total,
        'n_modes': n_modes,
        'n_modes_cosmic': n_modes_cosmic,
        'ell_from_k': ell_from_k,
        'Cl_approx': Cl_approx,
        'Dl_dimensionless': Dl_dimensionless,
        'Dl_uK2': Dl_uK2,
        'Dl_uK2_err_sample': Dl_uK2_err_sample,
        'Dl_uK2_err_cosmic': Dl_uK2_err_cosmic,
        'Dl_uK2_err_total': Dl_uK2_err_total,
        'hii_dim': hii_dim
    }

print("\n" + "="*70)
print("POWER SPECTRUM CALCULATION COMPLETE")
print(f"Computed for {len(power_spectrum_results)} BOX_LEN values")
print("="*70)
# =============================================================================
# PLOT 1: P(k) with Error Bars - All Box Sizes
# =============================================================================

print("\n" + "="*70)
print("GENERATING POWER SPECTRUM PLOTS WITH ERROR BUDGET")
print("="*70)

cmap = mpl.cm.rainbow
norm = mpl.colors.Normalize(
    vmin=BOX_LEN_VALUES.min(),
    vmax=BOX_LEN_VALUES.max()
)

def plot_Pk_with_errors(ax):
    for box_len in sorted(power_spectrum_results.keys()):
        results = power_spectrum_results[box_len]
        k_centers = results['k_centers']
        P1d = results['P1d']
        P1d_err = results['P1d_err_total']
        
        valid = ~np.isnan(P1d) & (P1d > 0)
        color = cmap(norm(box_len))
        
        ax.errorbar(
            k_centers[valid],
            P1d[valid],
            yerr=P1d_err[valid],
            color=color,
            linewidth=2.0,
            alpha=0.7,
            marker='o',
            markersize=4,
            capsize=3,
            capthick=1,
            label=f'{box_len:.0f} Mpc'
        )
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$k$ [Mpc$^{-1}$]')
    ax.set_ylabel(r'$P(k)$ [Mpc$^{2}$]')
    ax.legend(loc='best', ncol=1, fontsize=10)

plot_name = "kSZ_Pk_boxsize_all_with_errors"

save_pdf_png(
    plot_Pk_with_errors,
    plot_dir,
    plot_name,
    title=rf'kSZ Power Spectrum $P(k)$ with Errors (Fixed Resolution = {CELL_SIZE_MPC:.2f} Mpc)'
)

print(f"✓ Saved: {plot_name}")


# =============================================================================
# PLOT 2: D_ℓ in μK² with Error Bars - THE MAIN RAINBOW RESULT!
# =============================================================================

def plot_Dl_with_errors(ax):
    for box_len in sorted(power_spectrum_results.keys()):
        results = power_spectrum_results[box_len]
        ell_from_k = results['ell_from_k']
        Dl_uK2 = results['Dl_uK2']
        Dl_err = results['Dl_uK2_err_total']
        
        valid = ~np.isnan(Dl_uK2) & (Dl_uK2 > 0) & (ell_from_k > 10)
        color = cmap(norm(box_len))
        
        ax.errorbar(
            ell_from_k[valid],
            Dl_uK2[valid],
            yerr=Dl_err[valid],
            color=color,
            linewidth=2.5,
            alpha=0.75,
            marker='s',
            markersize=5,
            capsize=3,
            capthick=1.5,
            zorder=10,
            label=f'{box_len:.0f} Mpc'
        )
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'Multipole $\ell$')
    ax.set_ylabel(r'$D_\ell$ [$\mu$K$^2$]')
    ax.legend(loc='best', ncol=1, framealpha=0.9, fontsize=10)

plot_name = "kSZ_Dl_uK2_boxsize_all_with_errors"

save_pdf_png(
    plot_Dl_with_errors,
    plot_dir,
    plot_name,
    title=rf'kSZ $D_\ell$ with Error Budget (Fixed Resolution = {CELL_SIZE_MPC:.2f} Mpc)'
)

print(f"✓ Saved: {plot_name} *** MAIN RAINBOW PLOT WITH ERRORS ***")



# =============================================================================
# Summary Table WITH ERRORS
# =============================================================================

print("\n" + "="*70)
print("POWER SPECTRUM SUMMARY WITH ERROR BUDGET")
print("="*70)
print(f"{'BOX[Mpc]':<10} {'HII_DIM':<8} {'Cell[Mpc]':<10} {'ℓ_range':<20} "
      f"{'Peak_Dl[μK²]':<14} {'ℓ_peak':<8} {'Err[%]':<8}")
print("-" * 100)

for box_len in sorted(power_spectrum_results.keys()):
    results = power_spectrum_results[box_len]
    valid = ~np.isnan(results['Dl_uK2']) & (results['Dl_uK2'] > 0) & (results['ell_from_k'] > 10)
    
    hii_dim = results['hii_dim']
    cell_size_mpc = box_len / hii_dim
    
    if np.any(valid):
        ell_min = results['ell_from_k'][valid].min()
        ell_max = results['ell_from_k'][valid].max()
        peak_idx = np.argmax(results['Dl_uK2'][valid])
        peak_dl = results['Dl_uK2'][valid][peak_idx]
        ell_peak = results['ell_from_k'][valid][peak_idx]
        peak_err = results['Dl_uK2_err_total'][valid][peak_idx]
        peak_err_pct = (peak_err / peak_dl) * 100
        
        print(f"{box_len:<10.0f} {hii_dim:<8} {cell_size_mpc:<10.3f} "
              f"[{ell_min:<6.0f}, {ell_max:<6.0f}]     "
              f"{peak_dl:<14.4e} {ell_peak:<8.0f} {peak_err_pct:<8.1f}")

print(f"\n=== ERROR BUDGET NOTES ===")
print(f"1. Errors include both sample variance (FFT binning) and cosmic variance")
print(f"2. Cosmic variance: δP/P ≈ 1/√N_modes, where N_modes ∝ k² Δk V")
print(f"3. Total error = √(σ_sample² + σ_cosmic²)")
print(f"4. Larger boxes have MORE cosmic variance (fewer modes at large scales)")
print(f"5. Error percentages shown are at peak D_ℓ")

print("\n" + "="*70)
print("ALL kSZ POWER SPECTRUM ANALYSIS COMPLETE WITH ERRORS!")
print("="*70)


COMPUTING kSZ POWER SPECTRA WITH ERROR BUDGET

=== CMB TEMPERATURE ===
T_CMB(z=0) = 2.725 K
T_CMB(z=5) = 2.725 K = 2725000.00 μK
Angular diameter distance: D_A = 1300.0 Mpc
Comoving distance to z=5: χ = 7800.0 Mpc

BOX_LEN = 200 Mpc, HII_DIM = 64 (Cell = 3.125 Mpc)

=== MAP PROPERTIES ===
Map size: 64 × 64 pixels
Physical size: 200.0 × 200.0 Mpc²
Pixel size: 3.125 Mpc/pixel
Map RMS: 1.5109e-05
Mean subtracted: -3.1764e-22

=== 2D POWER SPECTRUM ===
P(k) range: [1.7106e-43, 3.3985e-11] Mpc²

=== k-SPACE GRID ===
dk (fundamental): 0.031416 Mpc⁻¹
k range: [0.000000, 1.421723] Mpc⁻¹

=== AZIMUTHALLY AVERAGED P(k) WITH ERRORS ===
Valid k bins: 29 out of 34
P(k) range: [6.3002e-15, 1.2538e-11] Mpc²

=== ERROR BUDGET ===
Sample variance:  16.5% (mean)
Cosmic variance:  24.0% (mean)
Total variance:   30.1% (mean)
N_modes range:    0.8 - 1021.0

=== D_ℓ CONVERSION ===
Conversion factor: T_CMB²(z=5) = 7.4256e+12 μK²
D_ℓ range: [3.8302e-01, 1.1069e+01] μK²
D_ℓ total error: 30.1% (mean)

BOX_LEN 